In [10]:
# %%
import jax
import jax.numpy as jnp
import numpy as onp
%load_ext autoreload
%autoreload 2
import jax_sn

jax.config.update("jax_enable_x64", True)
import equinox as eqx
import warp as wp 
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop
from jax_sn.operators.transport_operator import TransportOperatorBatchedPallas


# %%
N_PER_DIM = 10

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=2) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[N_PER_DIM - 3, 2, 1],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, N_PER_DIM), (1.523, N_PER_DIM)],
        use_numpy=True,
        single_angle_transport_operator_cls = TransportOperatorBatchedPallas, single_angle_transport_operator_cls_kwargs = {"batch_size": 100}
    )

onp.random.seed(2042026)


base_mgop = _build_base_mgop()
b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# %%
from jax_sn.operators.transport_operator.transport_operator_warp import TransportOperatorSequentialWarpSingleAngle, TransportOperatorSequentialWarpMultipleAngles, TransportOperatorLevelsWarpSingleAngle, TransportOperatorLevelsWarpMultipleAngles
top_sa_levels = TransportOperatorLevelsWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle[0],0, batch_size=100, max_n_levels= None)
top_ma_levels = TransportOperatorLevelsWarpMultipleAngles(TransportOperatorLevelsWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle, 0, batch_size = 100, max_n_levels=None))



In [47]:
from jax_sn.operators.multi_group_operator.lower_triangular_solver import MultiGroupLowerTriangularBlockSolver, SolverDiagnostics
from jax_sn.operators.multi_group_operator.right_hand_side import IsotropicSourceSingleGroup, solve_multigroup_operator_rhs
import lineax as lx
jax.clear_caches()

solver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-10, 1e-10, max_steps = 200), SolverDiagnostics(print_single_group = True, print_sweep = False))

rhs = IsotropicSourceSingleGroup.from_operator(base_mgop, jnp.ones((base_mgop.n_elements, base_mgop.n_basis)))

In [51]:
solution_0 = solve_multigroup_operator_rhs(base_mgop, rhs, solver )

Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps


In [52]:
mgop_levels = eqx.tree_at(lambda op: op.transport_operator, base_mgop, top_ma_levels)

In [54]:
solution_1 = solve_multigroup_operator_rhs(mgop_levels, rhs, solver )

Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
